## 1. Introduction & Strategy
- Objective: To create a Hybrid Recommendation System by combining Collaborative Filtering (Item-Item Similarity) and Content-Based Filtering (TF-IDF on metadata).

**Collaborative Filtering (User-User / Item-Item based)**
- User-User: Recommends books based on the similarity between users (i.e., users who liked similar books). This approach works well when the system has a lot of user data.

- Item-Item: Recommends books based on similarities between items (i.e., books that have similar ratings from users). This method is typically more stable and often used in systems like Amazon’s recommendation engine.

**Output Example (Collaborative Filtering):**

- Book Title: "Book A Title"

- CF Score: 0.98 (calculated based on the similarity between the target user and other users' ratings)

- Source: Collaborative Filtering

**Content-Based Filtering (metadata similarity)**
- Metadata-based recommendations: Books are recommended based on the content similarity between book metadata such as title, author, genre, etc. This method doesn't require user interaction but instead looks at book features.

**Output Example (Content-Based Filtering):**
- Book Title: "Book B Title"
- CB Score: 0.85 (calculated using cosine similarity based on book metadata)
- Source: Content-Based Filtering



## 2. Choose a Hybridization Strategy
We can pick one or more strategies for hybridization:

**Blending (Weighted Average)**
- **Description:** Normalize the similarity scores from both collaborative filtering and content-based filtering, and combine them using a weighted average.

**Formula:**
            final_score=α×collab_score+(1−α)×content_score
Where 
α is the blending factor, for example, 0.5 or 0.7, which determines how much weight is given to collaborative filtering vs. content-based filtering.

**How to Implement:**

- Normalize both collaborative filtering and content-based scores between 0 and 1.

- Blend the scores using the weighted average formula. The α parameter can be tuned to balance the two approaches.

**Conditional Switching**
- **Description:** Switch between collaborative filtering and content-based filtering based on user data.
    - If the user has a large number of ratings (active user), prioritize collaborative filtering.
    - If the user has few or no ratings (cold-start), use content-based filtering, as collaborative filtering requires user interaction.

**Stacked Recommendations**
- **Description:** Return the top-N books from both models and merge them, ensuring there are no duplicates.

**How to Implement:**
- Sort recommendations from both models based on their scores.
- Combine the results and remove any duplicate books.

## Choosen Blending Logic 
- Blending Formula:
        final_scores= final_scores=α×collab_scores+(1−α)×content_scores
Where:
- alpha ∈ [0, 1]
    - If alpha closer to 1 → More collaborative filtering influence.
    - If alpha closer to 0 → More content-based filtering influence.

- Conditional Tagging:
   - Based on the value of alpha, the recommendation source is tagged as:
        - 'Collaborative' if alpha > 0.5
        - 'Content-Based' if alpha <= 0.5



In [1]:
# Import Libraries

import pandas as pd
import numpy as np
import re
import random
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler

In [2]:
# Load and Prepare Data

rating_df = pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Cleaned_Datasets\Ratings_Cleaned.csv")
book_df = pd.read_csv(r"C:\Users\HP\Desktop\Bookify\Database\Cleaned_Datasets\Books_Cleaned.csv")

In [3]:
# Clean text function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

# Apply cleaning
book_df['title'] = book_df['title'].apply(clean_text)
book_df['author'] = book_df['author'].apply(clean_text)
book_df['publisher'] = book_df['publisher'].apply(clean_text)

# Create combined field
book_df['combined'] = book_df['title'] + ' ' + book_df['author'] + ' ' + book_df['publisher']

In [4]:
ratings_df = rating_df.head(30000)
books_df = book_df.head(30000)

In [5]:
# Build Collaborative Filtering

# Create User-Item matrix
user_item_matrix = ratings_df.pivot_table(index='user_id', columns='book_id', values='ratings')

# Item-Item Cosine Similarity
item_similarity = cosine_similarity(user_item_matrix.fillna(0).T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

In [6]:
# Build Content-Based Filtering

# TF-IDF Vectorization
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(books_df['combined'])

# Content Cosine Similarity
content_similarity = cosine_similarity(tfidf_matrix)
content_similarity_df = pd.DataFrame(content_similarity, index=books_df['book_id'], columns=books_df['book_id'])

### Hybrid  Recommendeation Function

In [29]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import random

def hybrid_recommendation(
    user_id, 
    alpha=0.5, 
    top_n=5, 
    exploration=False, 
    exploration_frac=0.2, 
    verbose=True
):
    user_ratings = ratings_df[ratings_df['user_id'] == user_id]
    
    # Cold Start Handling
    cold_start = False
    if user_ratings.empty:
        cold_start = True
        if verbose:
            print(f"\n No ratings found for user {user_id}. Switching to pure Content-Based recommendations.")
    
    collab_scores = np.zeros(len(books_df))
    content_scores = np.zeros(len(books_df))

    if not cold_start:
        for _, rating in user_ratings.iterrows():
            book_id = rating['book_id']
            user_rating = rating['ratings']

            if book_id in item_similarity_df.columns:
                collab_scores += user_rating * item_similarity_df[book_id].reindex(books_df['book_id']).fillna(0).values

            if book_id in content_similarity_df.columns:
                content_scores += content_similarity_df[book_id].reindex(books_df['book_id']).fillna(0).values

        if len(user_ratings) > 0:
            collab_scores /= len(user_ratings)
            content_scores /= len(user_ratings)
    else:
        # For cold start, only content-based scores
        content_scores = np.random.rand(len(books_df))  # Random small noise if needed

    # Normalize
    scaler = MinMaxScaler()
    collab_scores_scaled = scaler.fit_transform(collab_scores.reshape(-1, 1)).flatten()
    content_scores_scaled = scaler.fit_transform(content_scores.reshape(-1, 1)).flatten()

    # Dynamic α adjustment: if collaborative filtering scores are weak
    cf_nonzero_ratio = np.count_nonzero(collab_scores_scaled) / len(collab_scores_scaled)
    if cf_nonzero_ratio < 0.3:  # If less than 30% CF signals, trust more on CB
        alpha = 0.3
    elif cf_nonzero_ratio > 0.7:  # If lots of CF signals, trust CF more
        alpha = 0.7
    else:
        alpha = 0.5

    if cold_start:
        alpha = 0  # Only content-based

    # Final blended score
    final_scores = alpha * collab_scores_scaled + (1 - alpha) * content_scores_scaled

    # Create dataframe
    recommendations = pd.DataFrame({
        'book_id': books_df['book_id'],
        'title': books_df['title'],
        'cf_score': collab_scores_scaled,
        'cb_score': content_scores_scaled,
        'final_score': final_scores
    })

    if not cold_start:
        recommendations = recommendations[~recommendations['book_id'].isin(user_ratings['book_id'])]

    recommendations = recommendations.sort_values('final_score', ascending=False)

    top_recommendations = recommendations.head(top_n)

    # --- Exploration: Inject random obscure books
    if exploration:
        n_explore = int(exploration_frac * top_n)
        obscure_books = recommendations.sample(n=n_explore, random_state=42)
        top_recommendations = pd.concat([top_recommendations, obscure_books]).drop_duplicates('book_id')
        top_recommendations = top_recommendations.sort_values('final_score', ascending=False).head(top_n)

    if verbose:
        print(f"\n Top {top_n} Book Recommendations for User {user_id}:")
        display(top_recommendations[['title', 'cf_score', 'cb_score', 'final_score']])

    return top_recommendations


### Sample Outputs & Analysis 

### Active User (Many Ratings)
- **Scenario:** An active user has many ratings, allowing the system to leverage collaborative filtering.

 **Steps:**
- Select a user with a substantial number of ratings in the dataset (e.g., user_id = 8).
- Call the hybrid recommender function with this user's ID.

In [30]:
# Active User (Many Ratings)
hybrid_recommendation(user_id=8, top_n=5,exploration=True)


Top 5 Book Recommendations for User 8:


,book_id,title,final_score,source
23312,0446605964,never street amos walker mysteries paperback,0.392404,Hybrid
4474,0671776975,a river runs through it and other stories and...,0.258390,Hybrid
24623,1558021256,the black moon,0.257481,Hybrid
24750,0061320684,the cunning of history,0.248287,Hybrid
17464,1551668157,about that man,0.247342,Hybrid


#### **Observations:**
- Good mix of mystery novels, short stories, history.
- Some CF signal from "About That Man" — collaborative boost.

### New/Inactive User (Cold-Start)
**Scenario:** A new or inactive user has little to no ratings, meaning the system will rely more on content-based filtering.

**Steps:**
- Select a user with few or no ratings (e.g., user_id = 1000, assuming it's a cold-start user with no ratings).

- Call the hybrid recommender function, and since no ratings are available, the system should fall back on content-based recommendations.

In [24]:
# 2. New/Inactive User (Cold Start - User ID 1000)
hybrid_recommendation(user_id=1000, top_n=5, exploration=True)


No ratings found for user 1000. Switching to pure Content-Based recommendations.

Top 5 Book Recommendations for User 1000:


,title,cf_score,cb_score,final_score,source
12867,madame bovary signet classics paperback,0.0,1.000000,1.000000,Hybrid
2232,enders game ender wiggins saga paperback,0.0,0.999994,0.999994,Hybrid
10696,the curse of cain the untold story of john wil...,0.0,0.999950,0.999950,Hybrid
20241,the autobiography of saint therese of lisieux ...,0.0,0.999947,0.999947,Hybrid
7726,miss zukas shelves the evidence miss zukas mys...,0.0,0.999934,0.999934,Hybrid


,book_id,title,cf_score,cb_score,final_score,source
12867,0451528204,madame bovary signet classics paperback,0.0,1.000000,1.000000,Hybrid
2232,0812550706,enders game ender wiggins saga paperback,0.0,0.999994,0.999994,Hybrid
10696,1580060218,the curse of cain the untold story of john wil...,0.0,0.999950,0.999950,Hybrid
20241,0385029039,the autobiography of saint therese of lisieux ...,0.0,0.999947,0.999947,Hybrid
7726,0380804743,miss zukas shelves the evidence miss zukas mys...,0.0,0.999934,0.999934,Hybrid


#### **Observations:**
- Strong CB matches based on genre: mysteries, adventure, family drama.
- High cb_score because no CF history.



### Popular Books
**Scenario:** Test the system with popular books (high number of ratings), to ensure they are recommended to users (both active and cold-start).

**Steps**
- Identify a popular book (e.g., a book with a large number of ratings in the dataset).

- Test the hybrid recommender on both active and cold-start users to see if popular books are recommended.

In [10]:
# 3. Active User (User ID 9) + Popular Book Scenario
hybrid_recommendation(user_id=9, top_n=5, exploration=True)


 Top 5 Book Recommendations for User 9:


,title,cf_score,cb_score,final_score,source
6382,beloved,0.000000,0.850640,0.595448,Content
16821,the bluest eye,0.256487,0.550272,0.462136,Hybrid
12474,paradise,0.000000,0.656082,0.459257,Content
29456,beloved,0.000000,0.604946,0.423462,Content
3488,sula,0.000000,0.594770,0.416339,Content


,book_id,title,cf_score,cb_score,final_score,source
6382,0452280621,beloved,0.000000,0.850640,0.595448,Content
16821,0452282195,the bluest eye,0.256487,0.550272,0.462136,Hybrid
12474,0452280397,paradise,0.000000,0.656082,0.459257,Content
29456,8440656955,beloved,0.000000,0.604946,0.423462,Content
3488,0452283868,sula,0.000000,0.594770,0.416339,Content


#### **Observations:**

- Very Toni Morrison focused (Beloved, Bluest Eye, Sula).

- User 9 clearly likes literary African-American fiction.

- Two editions of Beloved — maybe suppress duplicates later.

In [11]:
# 4. New User (User ID 12) + Popular Book Scenario
hybrid_recommendation(user_id=12, top_n=5, exploration=True)


 Top 5 Book Recommendations for User 12:


,title,cf_score,cb_score,final_score,source
17560,what i wish id known before i got married,0.0,0.252346,0.176642,Content
12383,ten things i wish id known before i went out i...,0.0,0.245510,0.171857,Content
29998,25 stupid mistakes you dont want to make in th...,0.0,0.197452,0.138216,Content
21151,never call your broker on monday and 300 other...,0.0,0.187017,0.130912,Content
22354,buried mistakes,0.0,0.180455,0.126319,Content


,book_id,title,cf_score,cb_score,final_score,source
17560,1576737810,what i wish id known before i got married,0.0,0.252346,0.176642,Content
12383,0446526126,ten things i wish id known before i went out i...,0.0,0.245510,0.171857,Content
29998,0737306173,25 stupid mistakes you dont want to make in th...,0.0,0.197452,0.138216,Content
21151,0062701649,never call your broker on monday and 300 other...,0.0,0.187017,0.130912,Content
22354,0451403053,buried mistakes,0.0,0.180455,0.126319,Content


**Observations:**
- Heavy into self-help, life advice, mistakes to avoid.
- CB scores dominate.

### Obscure Books
**Scenario:** Test the system with obscure books (low number of ratings), to see if the system can still provide recommendations for less popular books.

**Steps:**
- Identify obscure books (those with few ratings or low popularity) in the dataset.
- Test the hybrid recommender with both active and cold-start users to see if obscure books are still recommended.

In [12]:
# 5. Active User (User ID 8) + Obscure Book Scenario
hybrid_recommendation(user_id=8, top_n=5, exploration=True)


 Top 5 Book Recommendations for User 8:


,title,cf_score,cb_score,final_score,source
23312,never street amos walker mysteries paperback,0.000000,0.560577,0.392404,Content
4474,a river runs through it and other stories and...,0.000000,0.369129,0.258390,Content
24623,the black moon,0.000000,0.367831,0.257481,Content
24750,the cunning of history,0.000000,0.354696,0.248287,Content
17464,about that man,0.115689,0.303764,0.247342,Hybrid


,book_id,title,cf_score,cb_score,final_score,source
23312,0446605964,never street amos walker mysteries paperback,0.000000,0.560577,0.392404,Content
4474,0671776975,a river runs through it and other stories and...,0.000000,0.369129,0.258390,Content
24623,1558021256,the black moon,0.000000,0.367831,0.257481,Content
24750,0061320684,the cunning of history,0.000000,0.354696,0.248287,Content
17464,1551668157,about that man,0.115689,0.303764,0.247342,Hybrid


In [13]:
# 6. New/Inactive User (Cold Start - User ID 20)
hybrid_recommendation(user_id=20, top_n=5, exploration=True)


 No ratings found for user 20. Switching to pure Content-Based recommendations.

 Top 5 Book Recommendations for User 20:


,title,cf_score,cb_score,final_score,source
9027,omerta,0.0,1.000000,1.000000,Content
21910,la nouvelle pornographie,0.0,0.999914,0.999914,Content
28382,the magic of recluce recluce series book 1,0.0,0.999812,0.999812,Content
4422,vida sin condiciones,0.0,0.999740,0.999740,Content
6336,caddie woodlawn,0.0,0.999682,0.999682,Content


,book_id,title,cf_score,cb_score,final_score,source
9027,0345432401,omerta,0.0,1.000000,1.000000,Content
21910,2070757811,la nouvelle pornographie,0.0,0.999914,0.999914,Content
28382,0812505182,the magic of recluce recluce series book 1,0.0,0.999812,0.999812,Content
4422,846630620X,vida sin condiciones,0.0,0.999740,0.999740,Content
6336,0020418809,caddie woodlawn,0.0,0.999682,0.999682,Content


**Observations:**
- Fantasy + classic mystery + literature blend.
- Strong CB relevance.

### Final Comments:
- Diversity: Yes (different genres: mystery, fantasy, historical fiction, self-help).
- Relevance: Matches user interests based on scoring logic.

- Duplication: Minor duplication (like 2 editions of Beloved); can suppress later if needed.

- Cold Start handled properly by switching to pure content-based.

